In [ ]:
import pandas as pd
import numpy as np
from snowflake.snowpark.context import get_active_session

session = get_active_session()

In [ ]:
N_ROWS = 1000

transport_mock_raw = session.sql(f"""
    WITH lookup_numbered AS (
        SELECT
            code,
            fclass,
            ROW_NUMBER() OVER (ORDER BY code) - 1 AS lookup_idx
        FROM VALUES
            (5601,'railway_station'),(5602,'railway_halt'),(5603,'tram_stop'),
            (5621,'bus_stop'),(5622,'bus_station'),(5641,'taxi_rank'),
            (5651,'airport'),(5652,'airfield'),(5655,'helipad'),
            (5656,'apron'),(5661,'ferry_terminal'),(5671,'aerialway_station')
        AS t(code, fclass)
    ),
    raw AS (
        SELECT
            ABS(RANDOM() % 9000000) + 1000000                        AS osm_id,
            ABS(RANDOM() % 12)                                       AS lookup_idx,
            ROUND(UNIFORM(-2.72::FLOAT, -2.48::FLOAT, RANDOM()), 5) AS lon,
            ROUND(UNIFORM(51.40::FLOAT, 51.55::FLOAT, RANDOM()), 5) AS lat
        FROM TABLE(GENERATOR(ROWCOUNT => {N_ROWS}))
    ),
    joined AS (
        SELECT r.osm_id, l.code, l.fclass, r.lon, r.lat
        FROM raw r
        JOIN lookup_numbered l ON r.lookup_idx = l.lookup_idx
    )
    SELECT
        osm_id,
        code,
        fclass,
        CASE
            WHEN fclass IN ('railway_station','bus_station','airport','ferry_terminal','bus_stop')
             AND UNIFORM(0::FLOAT, 1::FLOAT, RANDOM()) > 0.4
            THEN
                CASE ABS(RANDOM() % 8)
                    WHEN 0 THEN 'North ' WHEN 1 THEN 'South '
                    WHEN 2 THEN 'East '  WHEN 3 THEN 'West '
                    WHEN 4 THEN 'Upper ' WHEN 5 THEN 'Lower '
                    WHEN 6 THEN 'Old '   ELSE        'New '
                END ||
                CASE ABS(RANDOM() % 8)
                    WHEN 0 THEN 'Bristol '    WHEN 1 THEN 'Clifton '
                    WHEN 2 THEN 'Redland '    WHEN 3 THEN 'Bedminster '
                    WHEN 4 THEN 'Bishopston ' WHEN 5 THEN 'Horfield '
                    WHEN 6 THEN 'Totterdown ' ELSE        'Easton '
                END ||
                INITCAP(REPLACE(fclass, '_', ' '))
            ELSE NULL
        END AS name,
        TO_GEOGRAPHY(
            'MULTIPOLYGON (((' ||
                lon         || ' ' || lat         || ', ' ||
                (lon+0.001) || ' ' || lat         || ', ' ||
                (lon+0.001) || ' ' || (lat+0.001) || ', ' ||
                lon         || ' ' || (lat+0.001) || ', ' ||
                lon         || ' ' || lat         ||
            ')))'
        ) AS geometry
    FROM joined
""")
print(f"✓ Mock transport data stored in transport_mock_raw ({transport_mock_raw.count()} rows)")
transport_mock_raw.show(5)

In [ ]:
from snowflake.snowpark import functions as F
from snowflake.snowpark.functions import uniform, random, col, lit, when

transport_mock_dirty = transport_mock_raw.select(
    col("OSM_ID"),
    when(uniform(lit(0).cast("float"), lit(1).cast("float"), random()) < lit(0.015), lit(9999))
        .otherwise(col("CODE")).alias("CODE"),
    when(uniform(lit(0).cast("float"), lit(1).cast("float"), random()) < lit(0.03), lit("unknown_class"))
        .otherwise(col("FCLASS")).alias("FCLASS"),
    when(uniform(lit(0).cast("float"), lit(1).cast("float"), random()) < lit(0.02), lit(""))
        .otherwise(col("NAME")).alias("NAME"),
    col("GEOMETRY")
)
print(f"✓ Dirty transport data stored in transport_mock_dirty ({transport_mock_dirty.count()} rows)")

In [ ]:
fclass_lookup = {
    5601: "railway_station",  5602: "railway_halt",      5603: "tram_stop",
    5621: "bus_stop",         5622: "bus_station",        5641: "taxi_rank",
    5651: "airport",          5652: "airfield",           5655: "helipad",
    5656: "apron",            5661: "ferry_terminal",     5671: "aerialway_station",
}

df = transport_mock_dirty.select("OSM_ID", "CODE", "FCLASS", "NAME").to_pandas()
df.columns = df.columns.str.lower()
print("Loaded shape:", df.shape)

In [ ]:
print("Null counts:")
print(df.isnull().sum())

print("\nEmpty string names:")
print((df["name"].str.strip() == "").sum())

print("\nDuplicate osm_ids")
dupes = df[df.duplicated(subset="osm_id", keep=False)]
print(f"{len(dupes)} rows with duplicate osm_id")

In [ ]:
def is_valid(row):
    if row["code"] not in fclass_lookup:
        return False
    if row["fclass"] != fclass_lookup[row["code"]]:
        return False
    if fclass_lookup.get(row["code"]) and row["code"] != {v: k for k, v in fclass_lookup.items()}.get(row["fclass"]):
        return False
    return True

before = len(df)
df = df[df.apply(is_valid, axis=1)].reset_index(drop=True)
after = len(df)

print(f"Removed {before - after} rows with code/fclass mismatches")
print(f"Remaining rows: {after}")

In [ ]:
cleaned_df = df.copy()
cleaned_df["name"] = cleaned_df["name"].str.strip().replace("", pd.NA)

# Drop duplicate osm_ids, keeping first occurrence
before_dedup = len(cleaned_df)
cleaned_df = cleaned_df.drop_duplicates(subset="osm_id", keep="first").reset_index(drop=True)
after_dedup = len(cleaned_df)

print(f"Removed {before_dedup - after_dedup} duplicate osm_id rows")
print(f"Cleaned {len(cleaned_df)} rows")
print(f"Names with values: {cleaned_df['name'].notna().sum()}")
print(f"Names missing: {cleaned_df['name'].isna().sum()}")

In [ ]:
session.use_schema("SILVER")
output_table = f"silver_landuse"
cleaned_df.write.mode("overwrite").save_as_table(output_table)
print(f"Saved to AIRBNB_INVESTMENT_APP.SILVER.{output_table}")